# Visualization & Diagnostics

History matching produces a lot to look at — emulator fits, the shrinking
plausible region, how close the simulated outputs sit to their targets. This
notebook tours the built-in plotting and display API on the stochastic SIR model
used throughout these tutorials.

Every `plot_*` method returns Matplotlib axes and accepts an `ax=` argument, so
the figures render inline and compose into your own layouts. The same figures are
written to the engine's `output_dir` after each wave.

Each figure is explained inline as we go; the [plotting API reference](../api.md#plotting)
lists every `plot_*` function.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import historymatching as hm
from model import SIR, generate_observed_data

%matplotlib inline
np.random.seed(0)

## Set up and run a calibration

We recover the transmission/recovery rates of a synthetic outbreak, then use the
result to demonstrate each plot. (See `01_basic_workflow.ipynb` for the workflow
itself in detail.)

In [ ]:
beta_true, gamma_true = 1.3, 0.5
population_size, n_seed = 10_000, 100
incidence_obs, _ = generate_observed_data(beta_true=beta_true, gamma_true=gamma_true,
                                          population_size=population_size,
                                          n_seed_infections=n_seed)

def sir_sim(samples: pd.DataFrame) -> pd.DataFrame:
    df = samples.copy()
    if "rand_seed" not in df.columns:
        df["rand_seed"] = np.random.default_rng(0).integers(0, 2**31, size=len(df))
    rows = []
    for _, row in df.iterrows():
        model = SIR(beta=row["beta"], gamma=row["gamma"],
                    s0=population_size - n_seed, i0=n_seed, seed=int(row["rand_seed"]))
        inc = model.get_incidence()
        rec = {f"incidence_{i}": inc[i] for i in range(len(inc))}
        rec["peak_incidence"] = inc.max()
        rec["total_cases"] = inc.sum()
        rows.append(rec)
    return pd.DataFrame(rows)

observations = {
    "peak_incidence": (incidence_obs.max(), 50),
    "total_cases":    (incidence_obs.sum(), 200),
    "incidence_10":   (incidence_obs.iloc[10], 40),
}

engine = hm.HistoryMatching(
    bounds={"beta": (0.5, 3.0), "gamma": (0.1, 1.0)},
    observations=observations,
    function=sir_sim,
    emulator_type="gpr",
    n_samples=300,
    max_iterations=3,
    random_seed=123,
    feature_selection=["peak_incidence", "incidence_10"],
    output_dir=None,  # focus on interactive plots; no files written here
)
results = engine.run()
print(f"Completed {len(results)} waves.")

## Text summaries

`engine.get_status_summary()` folds the per-wave convergence and the surviving parameter
ranges into one report. `engine.nroy_summary()` returns the same per-parameter
information as a tidy DataFrame.

In [ ]:
print(engine.get_status_summary())

In [ ]:
engine.nroy_summary()

## Convergence

The NROY fraction is the share of fresh prior samples passing *all* emulator
constraints so far. A steadily falling fraction means the calibration is
successfully ruling out parameter space.

In [ ]:
engine.plot_convergence();

## The NROY parameter cloud

The headline result: a corner plot of the non-implausible region — marginals on
the diagonal, pairwise scatter below. We overlay the known true values with
`truth=`.

In [ ]:
truth = {"beta": beta_true, "gamma": gamma_true}
engine.plot_nroy(truth=truth);

`plot_marginals` shows just the one-dimensional posteriors, with the sample
median and the true value marked.

In [ ]:
engine.plot_marginals(truth=truth);

## Z-scores vs targets

For every target, the distribution of `(simulated - target) / target_std` across
the NROY samples, by wave. Bands inside the green ±threshold region and centred
on zero are consistent with the data.

In [ ]:
engine.plot_zscores();

## Constrained directions

A PCA view of which *combinations* of parameters the data constrained most —
useful for spotting non-identifiable directions that marginal plots miss.

In [ ]:
engine.plot_constrained_dims(n_top=2);

## Emulator quality

The emulators are only trustworthy if they reproduce the simulator. Inspect the
fit per feature for the final wave.

In [ ]:
final = results[-1]
final.quality_table()

In [ ]:
final.plot_emulator_quality();

In [ ]:
final.plot_predicted_vs_actual("peak_incidence");

## Ensemble fan plot vs observed data

History matching constrains *parameters*; to check the *outputs*, re-run the
simulator at NROY parameter sets and compare the trajectories to the observed
data. `hm.HistoryMatching.plot_ensemble_fan` is model-agnostic — give it a 2-D array of
trajectories and an observed series.

In [ ]:
nroy = engine.get_nroy_samples(40, method="lhs")  # unbiased draw
trajectories = np.array([
    SIR(beta=r["beta"], gamma=r["gamma"], s0=population_size - n_seed, i0=n_seed).get_incidence()
    for _, r in nroy.iterrows()
])
hm.HistoryMatching.plot_ensemble_fan(trajectories, observed=incidence_obs.values,
                     xlabel="Day", ylabel="Daily incidence",
                     title="NROY trajectories vs observed");

## Domain-object views

`ParameterSpace`, `ObservationData`, and `EmulatorBank` each have a `summary()`
and (where it makes sense) a plot.

In [ ]:
print(engine.parameter_space.summary())
print()
print(engine.observations.summary())
print()
print(engine.emulator_bank.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
engine.parameter_space.plot_bounds(reference=engine.parameter_space, ax=axes[0])
engine.observations.plot_targets(ax=axes[1])
plt.tight_layout();

All of these methods return Matplotlib axes, so you can drop them into your
own multi-panel figures (as above) or save them with `fig.savefig(...)`. For the
full catalogue of plotting functions, see the
[plotting API reference](../api.md#plotting).